# 04 - Sales Analysis

## Problem 3: Sales Performance Analysis

This notebook uses **Databricks SQL cells** for the sales analysis.

The capstone requires SQL queries to:
- Analyze sales trends across stores
- Identify best-selling products
- Compare store performance
- Use JOINs, GROUP BY, and aggregate functions

## Q1. Which store has the highest total revenue in 2024?

We join the sales table with the stores table and calculate total revenue for each store.

In [0]:
%sql

SELECT
    s.store_id,
    s.store_name,
    SUM(sa.total_amount) AS total_revenue
FROM sales_silver sa
INNER JOIN stores_silver s
    ON sa.store_id = s.store_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY
    s.store_id,
    s.store_name
ORDER BY total_revenue DESC
LIMIT 1;

## Q2. What are the top 5 best-selling products by quantity?

A product's sales volume is measured using the total quantity sold.

In [0]:
SELECT
    s.product_id,
    p.product_name,
    SUM(s.quantity_sold) AS total_quantity_sold
FROM sales_silver s
JOIN products_silver p
    ON s.product_id = p.product_id
GROUP BY
    s.product_id,
    p.product_name
ORDER BY total_quantity_sold DESC
LIMIT 5;

## Q3. How does each store rank by revenue and transaction count?

First, calculate revenue and transaction count for each store. Then use `DENSE_RANK()` to rank stores on both metrics.

In [0]:
SELECT
    st.store_id,
    st.store_name,
    SUM(s.total_amount) AS total_revenue,
    COUNT(DISTINCT s.transaction_id) AS transaction_count,

    DENSE_RANK() OVER (
        ORDER BY SUM(s.total_amount) DESC
    ) AS revenue_rank,

    DENSE_RANK() OVER (
        ORDER BY COUNT(DISTINCT s.transaction_id) DESC
    ) AS transaction_count_rank

FROM sales_silver s
JOIN stores_silver st
    ON s.store_id = st.store_id
WHERE YEAR(s.sale_date) = 2024
GROUP BY
    st.store_id,
    st.store_name
ORDER BY revenue_rank;

## Q4. Which products are most profitable per store?

### Important limitation

The available sales/product data does not contain product cost or profit margin.

Therefore, true profitability cannot be calculated.

Instead, we identify the **top 5 products by revenue for each store**. This is a revenue analysis, not a profit calculation.

In [0]:
%sql

WITH product_store_revenue AS (
    SELECT
        s.store_id,
        s.store_name,
        p.product_id,
        p.product_name,
        p.category,
        SUM(sa.quantity_sold) AS total_quantity_sold,
        ROUND(SUM(sa.total_amount), 2) AS total_revenue
    FROM sales_silver sa
    INNER JOIN stores_silver s
        ON sa.store_id = s.store_id
    INNER JOIN products_silver p
        ON sa.product_id = p.product_id
    WHERE YEAR(sa.sale_date) = 2024
    GROUP BY
        s.store_id,
        s.store_name,
        p.product_id,
        p.product_name,
        p.category
),

ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY store_id
            ORDER BY total_revenue DESC
        ) AS product_revenue_rank
    FROM product_store_revenue
)

SELECT
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    total_revenue,
    product_revenue_rank
FROM ranked_products
WHERE product_revenue_rank <= 5
ORDER BY
    store_id,
    product_revenue_rank;

## Q5. What is the month-on-month sales trend for each store?

Sales are grouped by store and month. The result includes monthly revenue, transaction count, and quantity sold.

In [0]:
%sql

SELECT
    s.store_id,
    s.store_name,
    DATE_FORMAT(sa.sale_date, 'yyyy-MM') AS sales_month,
    ROUND(SUM(sa.total_amount), 2) AS monthly_revenue,
    COUNT(DISTINCT sa.transaction_id) AS transaction_count,
    SUM(sa.quantity_sold) AS quantity_sold
FROM sales_silver sa
INNER JOIN stores_silver s
    ON sa.store_id = s.store_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY
    s.store_id,
    s.store_name,
    DATE_FORMAT(sa.sale_date, 'yyyy-MM')
ORDER BY
    s.store_id,
    sales_month;

## Q6. Which store has the most diverse product sales?

Product diversity is measured as the number of distinct products sold by each store.

In [0]:
%sql

SELECT
    s.store_id,
    s.store_name,
    COUNT(DISTINCT sa.product_id) AS unique_products_sold
FROM sales_silver sa
INNER JOIN stores_silver s
    ON sa.store_id = s.store_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY
    s.store_id,
    s.store_name
ORDER BY
    unique_products_sold DESC,
    s.store_id;


# Problem 3 Complete

The six required sales-analysis questions have been implemented using Databricks SQL.

### SQL concepts used

- `INNER JOIN`
- `GROUP BY`
- `SUM()`
- `COUNT()`
- `COUNT(DISTINCT)`
- `ORDER BY`
- `LIMIT`
- CTEs using `WITH`
- `DENSE_RANK()`
- `ROW_NUMBER()`

The SQL queries in this notebook can also be copied into the required `sales_analysis.sql` deliverable.